# 04 - LLM-Assisted Extraction

**Goal:** Use a local Large Language Model (Llama 3.2 3B) to intelligently parse OCR text.

**Why LLMs?** Rules are brittle. LLMs understand context (e.g., distinguishing a "quantity" from a "price" based on column position).

**Prerequisites:**
1. [Ollama](https://ollama.com/) installed and running.
2. Model pulled: `ollama pull llama3.2`

## Setup

In [18]:
import json
import os
import pandas as pd
from pathlib import Path
import ollama

# Paths
OUT_DIR = Path("../out")
OCR_JSON_FILES = list(OUT_DIR.glob("ocr_*.json"))

print(f"Found {len(OCR_JSON_FILES)} OCR result files")

Found 1 OCR result files


## Step 1: Check Environment

We need to make sure Ollama is reachable and the model exists.

In [19]:
MODEL_NAME = "llama3.2"

try:
    # Check available models
    models_response = ollama.list()
    
    # Handle response structure (object with .models attribute or dict)
    if hasattr(models_response, 'models'):
        model_list = models_response.models
    else:
        model_list = models_response.get('models', [])
    
    # Extract model names
    model_names = []
    for m in model_list:
        if hasattr(m, 'model'):
            model_names.append(m.model)
        elif isinstance(m, dict):
            model_names.append(m.get('model') or m.get('name'))
            
    print(f"Found models: {model_names}")
    
    if any(MODEL_NAME in m for m in model_names):
        print(f"✅ Model '{MODEL_NAME}' found locally.")
    else:
        print(f"⚠️ Model '{MODEL_NAME}' not found. Pulling it now... (this takes a moment)")
        os.system(f"ollama pull {MODEL_NAME}")
        print(f"✅ Model '{MODEL_NAME}' pulled.")
        
except Exception as e:
    print(f"❌ Could not connect to Ollama. Is it running? Error: {e}")
    print("Run 'ollama serve' in a separate terminal.")

Found models: ['llama3.2:latest', 'my-python-assistant:latest', 'codellama:7b-instruct', 'llama3:latest']
✅ Model 'llama3.2' found locally.


## Step 2: Prepare Input Data

We feed the LLM the raw text lines. To help it understand layout, we can sort them by Y-coordinate.

In [20]:
def load_ocr_text(json_path):
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    results = data.get('results', [])
    if not results: return ""
    
    # Sort by Y-center to preserve reading order
    # If 'center_y' key exists, use it to sort.
    if results and 'center_y' in results[0]:
        results.sort(key=lambda x: x['center_y'])
            
    # Create a single string representation
    text_lines = [item['text'] for item in results]
    return "\n".join(text_lines)

# Load first file
if OCR_JSON_FILES:
    json_path = OCR_JSON_FILES[0]
    ocr_text = load_ocr_text(json_path)
    print(f"Input Text ({len(ocr_text)} chars):\n---\n{ocr_text[:200]}...\n---")

Input Text (1360 chars):
---
Walmart
@
Save money. Live better.
3991
-
330
∞
339
(
DIANA EARNEST
MANAGER
231 BLUEBELL DR SW
PHILADELPHIA OH 44663
NEW
01301
ST#
44 TR#
02115
TE#
009044
OP#
7
1.97
PET
TOY
004747571658
1.97
FLOPPY
0...
---


## Step 3: Define Prompt & Schema

The key to getting JSON from an LLM is to provide a specific schema.

In [21]:
SYSTEM_PROMPT = """
You are an expert receipt parser. 
Extract structured data from the provided receipt text.
Return JSON only. No markdown formatting. No explanation.

JSON Schema:
{
    "vendor": "string",
    "date": "YYYY-MM-DD",
    "total": number,
    "items": [
        {"name": "string", "qty": number or null, "price": number}
    ]
}
"""

def build_user_prompt(text):
    return f"""Receipt Text:
{text}

Extract the data into JSON."""

## Step 4: Run Extraction

We send the request to Ollama.

In [22]:
def extract_with_llm(text):
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_prompt(text)},
    ])
    
    return response['message']['content']

# Run extraction
print("⏳ Asking Llama 3.2 to parse receipt...")
llm_response = extract_with_llm(ocr_text)
print("✅ Done!")

⏳ Asking Llama 3.2 to parse receipt...
✅ Done!


## Step 5: Parse and Validate Output

In [23]:
# Helper to clean markdown block format usually returned by LLMs
def clean_json_string(s):
    s = s.strip()
    if s.startswith("```json"):
        s = s[7:]
    if s.startswith("```"):
        s = s[3:]
    if s.endswith("```"):
        s = s[:-3]
    return s.strip()

# Parse JSON
data = {} # Initialize to avoid NameError
try:
    clean_response = clean_json_string(llm_response)
    data = json.loads(clean_response)
    print(json.dumps(data, indent=2))
except json.JSONDecodeError:
    print("❌ Failed to parse JSON. Raw output:")
    print(llm_response)

{
  "vendor": "Walmart",
  "date": "07/28/17",
  "total": 98.21,
  "items": [
    {
      "name": "PET",
      "qty": null,
      "price": 1.97
    },
    {
      "name": "FLOPPY",
      "qty": null,
      "price": 1.97
    },
    {
      "name": "PUPPY",
      "qty": 1,
      "price": 4.97
    },
    {
      "name": "SSSUPREME S",
      "qty": 2.5,
      "price": 5.92
    },
    {
      "name": "SQUEAK",
      "qty": 3,
      "price": 3.77
    },
    {
      "name": "MUNCHY DMBEL",
      "qty": null,
      "price": 2.92
    },
    {
      "name": "PED",
      "qty": 1,
      "price": 0.5
    },
    {
      "name": "HNYMD",
      "qty": 1,
      "price": 3.98
    },
    {
      "name": "FRENCH DRSNG",
      "qty": null,
      "price": 1.98
    },
    {
      "name": "ORANGES",
      "qty": 7,
      "price": 5.47
    },
    {
      "name": "BABY CARROTS",
      "qty": null,
      "price": 1.48
    },
    {
      "name": "COLLARDS",
      "qty": null,
      "price": 1.24
    },
    {
   

## Step 6: Compare and Export

Let's compare results and save for the benchmark notebook.

In [24]:
# Load previous results
rules_df_path = OUT_DIR / "receipt_items.csv"
if rules_df_path.exists():
    rules_df = pd.read_csv(rules_df_path)
    print("--- Rules-Based Items ---")
    print(rules_df[['item_name', 'price']].head(8))
else:
    print("Run notebook 03 first to compare!")

print("\n--- LLM Extracted Items ---")
if 'items' in data:
    llm_df = pd.DataFrame(data['items'])
    if not llm_df.empty and 'name' in llm_df and 'price' in llm_df:
        print(llm_df[['name', 'price']])
        
        # Save to CSV for benchmarking
        llm_df.to_csv(OUT_DIR / "llm_receipt_items.csv", index=False)
        print(f"\n✅ Saved LLM results to {OUT_DIR}/llm_receipt_items.csv")
    else:
        print(llm_df)
else:
    print("No items found in LLM output.")

--- Rules-Based Items ---
                     item_name  price
0    PET TOY 004747571658 1.97   7.00
1    FLOPPY PUPPY 004747514846   1.97
2     SSSUPREME S 070060332153   4.97
3      2.5 SQUEAK 084699803238   5.92
4    MUNCHY DMBEL 068113108796   3.77
5       DOG TREAT 007119013654   2.92
6  HNYMD SMORES 088491226837 H   3.98
7  FRENCH DRSNG 004132100655 H   1.98

--- LLM Extracted Items ---
            name  price
0            PET   1.97
1         FLOPPY   1.97
2          PUPPY   4.97
3    SSSUPREME S   5.92
4         SQUEAK   3.77
5   MUNCHY DMBEL   2.92
6            PED   0.50
7          HNYMD   3.98
8   FRENCH DRSNG   1.98
9        ORANGES   5.47
10  BABY CARROTS   1.48
11      COLLARDS   1.24
12       CALZONE  19.77
13  STKOBRLPLABL   1.97
14       SUNFLWR   0.97
15         BEADS   0.97
16   GREAT VALUE   9.97
17        LIPTON   4.48
18       DRY DOG  12.44

✅ Saved LLM results to ../out/llm_receipt_items.csv


---

## 🎯 Conclusion

**Local LLMs** are powerful for:
1.  **Handling structural noise** (e.g., ignoring random numbers on the line).
2.  **Contextual understanding** (knowing that "TAX" is not an item).
3.  **Zero-shot flexibility** (no regex tuning needed).

**Trade-off:** Slower than Rules/Regex, but much more robust for complex documents.